# Análise de Desempenho de Algoritmos em Ambientes Concorrentes e Paralelos: Um Estudo Comparativo em Java
**Universidade de Fortaleza (UNIFOR) — Centro de Ciências Tecnológicas (CCT) — Ciência da Computação**

Este *Jupyter Notebook* complementa a aplicação gráfica em **Java Swing**, realizando o processamento estatístico e a geração de visualizações de alta resolução a partir dos arquivos CSV exportados pelo framework de testes em Java:
- `resultados/benchmark_resumo.csv`: Estatísticas consolidadas (Média, Desvio Padrão, Mediana, Mínimo, Máximo, Speedup e Eficiência).
- `resultados/benchmark_amostras.csv`: Medições individuais das 5 amostras coletadas para cada configuração.
- `resultados/benchmark_intervalos.csv`: Estudo do impacto da amplitude do intervalo de valores ($k$) em função do tempo.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#CBD5E1'
plt.rcParams['axes.linewidth'] = 0.8

df_resumo = pd.read_csv('resultados/benchmark_resumo.csv')
df_amostras = pd.read_csv('resultados/benchmark_amostras.csv')
df_intervalos = pd.read_csv('resultados/benchmark_intervalos.csv')

df_resumo['CV_Percentual'] = (df_resumo['DesvioPadraoMs'] / df_resumo['MediaMs']) * 100.0
df_resumo['Eficiencia_Percentual'] = df_resumo['Eficiencia'] * 100.0

print(f'Total de configurações avaliadas: {len(df_resumo)}')
print(f'Total de execuções individuais registradas: {len(df_amostras)}')
df_resumo.head(10)

## 1. Análise Estatística Descritiva (Conjunto Aleatório — $N = 100.000$ e $N = 1.000.000$)
Abaixo são apresentadas as métricas de tendência central (Média e Mediana), dispersão (Desvio Padrão e Coeficiente de Variação) e desempenho paralelo (Speedup $S_p = T_1 / T_p$ e Eficiência $E_p = S_p / p$).

In [ ]:
filtro_100k = df_resumo[(df_resumo['NaturezaDados'] == 'ALEATORIO') & (df_resumo['Tamanho'] == 100000)]
colunas_exibicao = ['Algoritmo', 'Modo', 'Threads', 'MediaMs', 'DesvioPadraoMs', 'MedianaMs', 'CV_Percentual', 'Speedup', 'Eficiencia_Percentual']
display(filtro_100k[colunas_exibicao].round(4))

filtro_1m = df_resumo[(df_resumo['NaturezaDados'] == 'ALEATORIO') & (df_resumo['Tamanho'] == 1000000)]
display(filtro_1m[colunas_exibicao].round(4))

## 2. Escalabilidade: Tempo de Execução em Função do Tamanho do Vetor ($N$)
Comparamos o comportamento assintótico dos quatro algoritmos (*Quick Sort*, *Merge Sort*, *Counting Sort* e *Bubble Sort*) em execução Serial (1 Thread) e Paralela (8 e 16 Threads).

In [ ]:
cores = {
    'Quick Sort': '#2563EB',
    'Merge Sort': '#10B981',
    'Counting Sort': '#F59E0B',
    'Bubble Sort': '#DC2626'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for idx, thr in enumerate([1, 8]):
    ax = axes[idx]
    sub = df_resumo[(df_resumo['NaturezaDados'] == 'ALEATORIO') & (df_resumo['Threads'] == thr)]
    for alg, grupo in sub.groupby('Algoritmo'):
        grupo = grupo.sort_values('Tamanho')
        ax.errorbar(
            grupo['Tamanho'], grupo['MediaMs'], yerr=grupo['DesvioPadraoMs'],
            label=alg, color=cores.get(alg, '#333'), marker='o', linewidth=2.2, capsize=4
        )
    modo_str = 'Serial (1 Thread)' if thr == 1 else f'Paralelo ({thr} Threads)'
    ax.set_title(f'Escalabilidade por Tamanho (N) — {modo_str}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Tamanho do Conjunto de Dados (N)')
    ax.set_ylabel('Tempo Médio de Execução (ms) — Escala Log')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.grid(True, which='both', linestyle='--', alpha=0.4)
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Análise de Speedup ($S_p$) e Eficiência Paralela ($E_p$) por Número de Threads
Investigação do ganho de desempenho ao variar a disponibilidade de núcleos de processamento ($1, 2, 4, 8, 16$ threads).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Speedup para N = 100.000 (inclui Bubble Sort) e N = 1.000.000 (algoritmos rápidos)
for idx, tam in enumerate([100000, 1000000]):
    ax = axes[idx]
    sub = df_resumo[(df_resumo['NaturezaDados'] == 'ALEATORIO') & (df_resumo['Tamanho'] == tam)]
    for alg, grupo in sub.groupby('Algoritmo'):
        grupo = grupo.sort_values('Threads')
        ax.plot(
            grupo['Threads'], grupo['Speedup'],
            label=alg, color=cores.get(alg, '#333'), marker='s', linewidth=2.3
        )
    ax.axhline(1.0, color='#94A3B8', linestyle='--', linewidth=1.2, label='Baseline Serial (1.0x)')
    ax.set_title(f'Curva de Speedup (Sp) × Threads (N = {tam:,})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Número de Threads (p)')
    ax.set_ylabel('Speedup (T_serial / T_paralelo)')
    ax.set_xticks([1, 2, 4, 8, 16])
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Impacto da Natureza dos Dados e do Intervalo de Valores ($k$)
Avaliamos como a distribuição inicial dos elementos (`ALEATORIO`, `INVERSAMENTE_ORDENADO`, `QUASE_ORDENADO`, `POUCOS_UNICOS`) e a variação da amplitude máxima de valores $k \in [1.000, 10.000.000]$ afetam cada algoritmo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Painel esquerdo: Natureza dos dados (N = 100.000, 8 threads, sem Bubble Sort para resolução visual)
ax1 = axes[0]
sub_nat = df_resumo[(df_resumo['Tamanho'] == 100000) & (df_resumo['Threads'] == 8) & (df_resumo['Algoritmo'] != 'Bubble Sort')]
pivot_nat = sub_nat.pivot(index='NaturezaDados', columns='Algoritmo', values='MediaMs')
pivot_nat.plot(kind='bar', ax=ax1, color=[cores.get(c, '#333') for c in pivot_nat.columns], rot=15)
ax1.set_title('Impacto da Natureza dos Dados (N = 100.000, 8 Threads)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Tempo Médio (ms)')
ax1.set_xlabel('Natureza do Conjunto de Dados')
ax1.grid(axis='y', linestyle='--', alpha=0.4)

# Painel direito: Intervalo de valores k em função do tempo (N = 500.000)
ax2 = axes[1]
sub_int = df_intervalos[df_intervalos['Threads'] == 1]
for alg, grupo in sub_int.groupby('Algoritmo'):
    grupo = grupo.sort_values('IntervaloMax')
    ax2.plot(
        grupo['IntervaloMax'], grupo['MediaMs'],
        label=f'{alg} (Serial)', color=cores.get(alg, '#333'), marker='o', linewidth=2.2
    )
sub_int_par = df_intervalos[df_intervalos['Threads'] == 8]
for alg, grupo in sub_int_par.groupby('Algoritmo'):
    grupo = grupo.sort_values('IntervaloMax')
    ax2.plot(
        grupo['IntervaloMax'], grupo['MediaMs'],
        label=f'{alg} (8 Threads)', color=cores.get(alg, '#333'), marker='^', linestyle='--', linewidth=2.0
    )
ax2.set_xscale('log')
ax2.set_title('Impacto do Intervalo de Valores [0..k] no Tempo (N = 500.000)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Limite Superior do Intervalo de Valores (k)')
ax2.set_ylabel('Tempo Médio de Execução (ms)')
ax2.grid(True, which='both', linestyle='--', alpha=0.4)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()